# Hinglish -> Mixed Script Converter  (v4)
**Model:** `qwen3:14b` via Ollama

Converts Roman Hinglish so that:
- **Pure Hindi grammar words** -> Devanagari (hai->है, tum->तुम, woh->वो, mein->में)
- **English-origin words** -> stay Latin (perfect, energy, murder, tragedy, Sherlock)

### Key fix in v4
Previous versions had a short KEEP IN LATIN whitelist, so the model was
transliterating English words like `perfect->परफेक्ट`, `energy->एनर्जी`.
v4 flips the default rule: **English-origin words ALWAYS stay Latin.**
Only pure Hindi grammatical words (postpositions, pronouns, verbs, conjunctions) go to Devanagari.

---
> Go to **Runtime > Change runtime type > T4 GPU**

## Step 1 - Install Ollama


In [1]:
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (470 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently i

## Step 2 - Start Server

In [8]:
import subprocess, time, requests, re, json

server = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print('Starting Ollama server', end='')
for _ in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            print(' Ready!')
            break
    except Exception:
        pass
    print('.', end='', flush=True)
    time.sleep(1)
else:
    print(' Did not start. Re-run this cell.')

Starting Ollama server Ready!


## Step 3 - Pull qwen3:14b
> First time: ~9.3 GB, ~8-12 minutes. Subsequent runs use cache.

In [9]:
MODEL = 'qwen3:14b'
print(f'Pulling {MODEL} (~9.3 GB)...')
result = subprocess.run(['ollama', 'pull', MODEL], capture_output=True, text=True)
if result.returncode == 0:
    print(f'{MODEL} ready!')
else:
    print('Pull failed:', result.stderr)

Pulling qwen3:14b (~9.3 GB)...
qwen3:14b ready!


## Step 4 - Verify GPU

In [10]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader 2>/dev/null \
    || echo 'No GPU - will use CPU (much slower)'

Tesla T4, 15360 MiB, 5492 MiB


## Step 5 - Converter Setup

### The core rule change in v4
The prompt now has **two tiers of rules** instead of a whitelist:

**Tier 1 — Convert to Devanagari:** Only pure Hindi grammatical words (postpositions, pronouns,
conjunctions, common Hindi-origin verbs and nouns).

**Tier 2 — Keep in Latin (the default):** Every English-origin word, even if used in Hinglish context.
This includes descriptive words (perfect, clear, sensitive), action words (share, murder),
and abstract nouns (tragedy, energy, emotion, ambition).

In [11]:
OLLAMA_CHAT_URL = 'http://localhost:11434/api/chat'

SYSTEM_PROMPT = (
    """Tu ek Hinglish-to-mixed-script converter hai.

## MAIN GOAL
Roman Hinglish text mein SIRF pure Hindi grammatical words ko Devanagari mein badal.
Har English-origin word -- chahe woh koi bhi ho -- Latin script mein REHNE DO.

## GOLDEN RULE
Agar ek word English dictionary mein milta hai, toh usse Latin mein rakho.
Sirf woh words Devanagari mein jao jo PURE HINDI grammatical words hain.
Transliterate mat karo. 'perfect' ko 'परफेक्ट' mat likho. 'energy' ko 'एनर्जी' mat likho.
'murder' ko 'मर्डर' mat likho. English words ENGLISH mein rahenge.

## ABSOLUTE RULES
1. Output mein SIRF converted text -- koi explanation ya reasoning BILKUL NAHI
2. Koi bhi word ADD ya REMOVE mat karo
3. Punctuation, line breaks, spacing EXACTLY preserve karo
4. 'the' (English article) -> KEEP IN LATIN, never convert to थे
5. Doubt ho toh LATIN mein rakho

## DEVANAGARI MEIN CONVERT KAR -- SIRF YEH WORDS

[ VERBS: TO BE / STATE -- pure Hindi ]
hai -> है | hain -> हैं | tha -> था | thi -> थी
hoga -> होगा | hogi -> होगी | honge -> होंगे
hota -> होता | hoti -> होती | hote -> होते
hua -> हुआ | hui -> हुई | hue -> हुए
ho -> हो | hun -> हूँ | hoon -> हूँ

[ PRONOUNS -- pure Hindi ]
main -> मैं | mai -> मैं | mujhe -> मुझे | mujhse -> मुझसे
mera -> मेरा | meri -> मेरी | mere -> मेरे
tum -> तुम | tumhara -> तुम्हारा | tumhari -> तुम्हारी | tumhare -> तुम्हारे | tumhe -> तुम्हें
woh -> वो | wo -> वो
uska -> उसका | uski -> उसकी | uske -> उसके | use -> उसे | usne -> उसने | usse -> उससे
yeh -> यह | ye -> ये
hum -> हम | humara -> हमारा | humari -> हमारी | humare -> हमारे | hume -> हमें | humne -> हमने
aap -> आप | aapka -> आपका | aapki -> आपकी | aapke -> आपके
unka -> उनका | unki -> उनकी | unke -> उनके | unhe -> उन्हें | unhone -> उन्होंने
khud -> खुद

[ POSTPOSITIONS -- pure Hindi grammar ]
mein -> में | se -> से | ko -> को | ne -> ने
ka -> का | ki -> की | ke -> के
pe -> पे | par -> पर | tak -> तक
ke liye -> के लिए | ke saath -> के साथ | ke baad -> के बाद
ke pehle -> के पहले | ke paas -> के पास | ke baare mein -> के बारे में
ke bina -> के बिना | ke upar -> के ऊपर | ke andar -> के अंदर

[ CONJUNCTIONS -- pure Hindi ]
aur -> और | ya -> या | lekin -> लेकिन | magar -> मगर
toh -> तो | kyunki -> क्योंकि | isliye -> इसलिए | agar -> अगर | jabki -> जबकि

[ ADVERBS -- pure Hindi ]
nahi -> नहीं | nahin -> नहीं | mat -> मत
bhi -> भी | hi -> ही | ab -> अब | phir -> फिर
sirf -> सिर्फ | bahut -> बहुत | bilkul -> बिल्कुल | zyada -> ज़्यादा
thoda -> थोड़ा | kaafi -> काफ़ी | zaroor -> ज़रूर | shayad -> शायद
hamesha -> हमेशा | kabhi -> कभी | pehle -> पहले | abhi -> अभी | jaldi -> जल्दी
seedha -> सीधा | achanak -> अचानक | phir bhi -> फिर भी

[ QUESTION WORDS -- pure Hindi ]
kya -> क्या | kyun -> क्यों | kyon -> क्यों
kaise -> कैसे | kaisa -> कैसा | kaisi -> कैसी
kaun -> कौन | kahan -> कहाँ | kab -> कब | yahan -> यहाँ | wahan -> वहाँ
kitna -> कितना | kitni -> कितनी | kitne -> कितने

[ COMMON HINDI VERBS -- only pure Hindi origin verbs ]
dekha -> देखा | dekhi -> देखी | dekhe -> देखे | dekho -> देखो
kaha -> कहा | bola -> बोला | boli -> बोली | bole -> बोले
gaya -> गया | gayi -> गई | gaye -> गए
aaya -> आया | aayi -> आई | aaye -> आए
raha -> रहा | rahi -> रही | rahe -> रहे
liya -> लिया | diya -> दिया | kiya -> किया | kiye -> किए
lagta -> लगता | lagti -> लगती | lagte -> लगते
chahiye -> चाहिए | chahta -> चाहता | chahti -> चाहती
jaanta -> जानता | jaanti -> जानती
karta -> करता | karti -> करती | karte -> करते
sunta -> सुनता | suno -> सुनो | suna -> सुना
chala -> चला | chali -> चली | chale -> चले
socha -> सोचा | ruka -> रुका | uthaya -> उठाया
nikla -> निकला | batao -> बताओ | bataya -> बताया
samjha -> समझा | samjhi -> समझी | jaata -> जाता
bol utha -> बोल उठा | bol uthi -> बोल उठी

[ PURE HINDI NOUNS -- only words with no common English equivalent in Hinglish ]
baat -> बात | baatein -> बातें | kaam -> काम
ghar -> घर | din -> दिन | raat -> रात | waqt -> वक्त
baar -> बार | log -> लोग | logo -> लोगों
dil -> दिल | duniya -> दुनिया | zindagi -> ज़िंदगी
aankh -> आँख | aankhein -> आँखें | aankhon -> आँखों
haath -> हाथ | chehra -> चेहरा | aawaz -> आवाज़
shaadi -> शादी | naukrani -> नौकरानी | kursi -> कुर्सी
darwaza -> दरवाज़ा | kamra -> कमरा | kamre -> कमरे
hafte -> हफ्ते | hafton -> हफ्तों
taraf -> तरफ | paas -> पास | andar -> अंदर | bahar -> बाहर

[ PURE HINDI ADJECTIVES ]
achha -> अच्छा | accha -> अच्छा | achi -> अच्छी | ache -> अच्छे
bura -> बुरा | buri -> बुरी
bada -> बड़ा | badi -> बड़ी | bade -> बड़े
chota -> छोटा | choti -> छोटी
naya -> नया | nayi -> नई | naye -> नए
purana -> पुराना | purani -> पुरानी | purane -> पुराने
khaas -> ख़ास | ajeeb -> अजीब | theek -> ठीक
bechain -> बेचैन | khush -> खुश | pareshan -> परेशान
puri -> पूरी | pura -> पूरा | saari -> सारी
aisa -> ऐसा | aisi -> ऐसी | aise -> ऐसे

[ FILLERS / EXPRESSIONS -- pure Hindi ]
yaar -> यार | dost -> दोस्त | haan -> हाँ | sach -> सच
ek -> एक | aaj -> आज | kal -> कल | saath -> साथ
koi -> कोई | kuch -> कुछ | sab -> सब | sabhi -> सभी
khwahish -> ख्वाहिश | taana -> ताना | salah -> सलाह
hadsa -> हादसा | marhoom -> मरहूम

## ENGLISH WORDS -- ALWAYS LATIN (examples, not exhaustive)
These and ALL similar English-origin words MUST stay in Latin script:
perfect, clear, share, murder, tragedy, energy, emotion, ambition,
sensitive, instrument, nature, admirable, questionable, singular,
reasoning, observing, machine, results, mental, distraction,
lodgings, practice, mission, family, companion, friend, former,
powerful, faculty, extraordinary, official, hopeless, daily,
complete, delicate, adjusted, temperament, balanced, precise, cold,
gender, passion, observer, motive, action, trained, crack, grit,
lens, natural, reigning, singular, vague, summoned, account,
cocaine, cocaine, iodoform, nitrate, gasogene, stethoscope,
Holmes, Watson, Irene, Adler, Mary, Jane, Sherlock, Bohemia,
Baker, Street, London, Odessa, Afghanistan, Netley, Trincomalee,
coat, desk, chair, table, bell, case, hotel, phone, doctor, police,
book, note, paper, stamp, mark, sign, record, post, letter,
service, club, class, type, form, kind, sort, style, mode

## CONVERSION EXAMPLES -- STUDY CAREFULLY

INPUT:  Sherlock Holmes ke liye woh hamesha THE woman rahi.
OUTPUT: Sherlock Holmes के लिए वो हमेशा THE woman रही.

INPUT:  Woh duniya ka sabse perfect reasoning machine tha.
OUTPUT: वो दुनिया का sabse perfect reasoning machine था.
NOTE:   'perfect', 'reasoning', 'machine' are English -- they stay Latin.
        'duniya' is pure Hindi -- Devanagari.
        'sabse' is Hindi superlative -- stays Latin (not in table, per Rule 5).

INPUT:  Uske liye strong emotions bahut disturbing hoti thi.
OUTPUT: उसके लिए strong emotions बहुत disturbing होती थी.
NOTE:   'emotions', 'strong', 'disturbing' are English -- they stay Latin.

INPUT:  Maine uske murder case ke baare mein suna tha.
OUTPUT: मैंने उसके murder case के बारे में सुना था.
NOTE:   'murder', 'case' are English -- they stay Latin.

INPUT:  Woh bohemian soul wala tha -- society ke har form se nafrat thi usse.
OUTPUT: वो bohemian soul वाला था -- society के हर form से nafrat थी उससे.
NOTE:   'bohemian', 'soul', 'society', 'form' are English -- they stay Latin.
        'nafrat' is Urdu/Hindi -- can stay Latin per Rule 5 (doubt).

INPUT:  Holmes kursi se ucha. Coat uthaya. Main kuch bolta -- usse pehle woh ja chuka tha.
OUTPUT: Holmes कुर्सी से उछा. Coat उठाया. मैं कुछ बोलता -- उससे पहले वो जा चुका था.

INPUT:  Shaadi tumhare liye achhi hai. Tumne saadhe saat pound weight badha liya hai.
OUTPUT: शादी तुम्हारे लिए अच्छी है. तुमने साढ़े सात pound weight बढ़ा लिया है."""
)

THINKING_PATTERNS = [
    r'^Okay,?\s*(let\'?s|I\'?ll|I need|I will)',
    r'^Let me\s+',
    r'^I need to\s+',
    r'^First,?\s+I',
    r'^Sure,?\s+',
    r'^Here(\'?s| is)\s+',
    r'^The user wants',
    r'^To convert\s+',
    r'^Looking at\s+',
]

def strip_think_tags(text):
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

def strip_reasoning_preamble(text):
    text = strip_think_tags(text)
    lines = text.split('\n')
    for i, line in enumerate(lines):
        s = line.strip()
        if not s:
            continue
        if not any(re.match(p, s, re.IGNORECASE) for p in THINKING_PATTERNS):
            return '\n'.join(lines[i:]).strip()
    return text.strip()

def validate_output(original, output):
    if len(output) > len(original) * 3:
        return False
    has_devanagari = bool(re.search(r'[\u0900-\u097F]', output))
    has_hindi = bool(re.search(
        r'\b(hai|hain|toh|nahi|aur|mein|tum|main|woh|kya|bhi)\b',
        original, re.IGNORECASE))
    if has_hindi and not has_devanagari:
        return False
    return True

def convert_chunk(text, retry=True):
    """
    /api/chat with think=False -- the ONLY correct Ollama API
    call that disables Qwen3 thinking mode.
    """
    user_msg = (
        'Output ONLY the converted text. Start with the first word directly.\n'
        'Remember: English-origin words like perfect, energy, murder, tragedy, '
        'emotions, clear, share, instrument, sensitive, reasoning etc. MUST stay in Latin.\n\n'
        + text
    )
    payload = {
        'model': MODEL,
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_msg}
        ],
        'stream': False,
        'think': False,
        'options': {
            'temperature': 0.0,
            'top_k': 1,
            'num_predict': 2048,
        }
    }
    response = requests.post(OLLAMA_CHAT_URL, json=payload, timeout=180)
    response.raise_for_status()
    raw = response.json()['message']['content'].strip()
    cleaned = strip_reasoning_preamble(raw)

    if retry and not validate_output(text, cleaned):
        print('  Retrying...', end=' ')
        payload['messages'][1]['content'] = (
            'IMPORTANT: Output ONLY the converted text. No explanation.\n\n' + text
        )
        r2 = requests.post(OLLAMA_CHAT_URL, json=payload, timeout=180)
        r2.raise_for_status()
        cleaned = strip_reasoning_preamble(r2.json()['message']['content'].strip())

    return cleaned

def process_in_chunks(full_text, chunk_size=350, verbose=True):
    paragraphs = full_text.split('\n')
    converted_parts = []
    buffer = []
    buffer_len = 0
    chunk_count = 0

    def flush_buffer():
        nonlocal chunk_count
        if not buffer:
            return
        chunk_text = '\n'.join(buffer)
        chunk_count += 1
        if verbose:
            print(f'  Chunk {chunk_count} ({len(chunk_text)} chars)...', end=' ', flush=True)
        result = convert_chunk(chunk_text)
        if verbose:
            print('done')
        converted_parts.append(result)
        buffer.clear()

    for para in paragraphs:
        if len(para.strip()) == 0:
            flush_buffer()
            converted_parts.append('')
            buffer_len = 0
        elif buffer_len + len(para) > chunk_size:
            flush_buffer()
            buffer.append(para)
            buffer_len = len(para)
        else:
            buffer.append(para)
            buffer_len += len(para)

    flush_buffer()
    if verbose:
        print(f'Done! {chunk_count} chunks processed.')
    return '\n'.join(converted_parts)

print('Converter v4 loaded.')
print('  Endpoint : /api/chat  |  think: False')
print(f'  Model    : {MODEL}')
print('  Mode     : English-origin words stay Latin (not transliterated)')

Converter v4 loaded.
  Endpoint : /api/chat  |  think: False
  Model    : qwen3:14b
  Mode     : English-origin words stay Latin (not transliterated)


## Step 6 - Quick Test
This test specifically checks that English words are NOT being transliterated.

In [13]:
tests = [
    # Test 1: English adjectives/nouns must stay Latin
    'Woh duniya ka sabse perfect reasoning machine tha, lekin emotions usse disturb karti thi.',
    # Test 2: English action words must stay Latin
    'Maine Atkinson brothers ki singular tragedy ke baare mein suna tha.',
    # Test 3: Mixed -- classic Holmes line
    'Holmes ke liye strong emotions uske sensitive instrument mein crack jaisi thi.',
    # Test 4: Confirm 'the' not converted to the
    'Sherlock Holmes ke liye woh hamesha THE woman rahi.',
]

# English words that should NOT appear in Devanagari
BAD_TRANSLITERATIONS = [
    ('perfect','परफेक्ट'), ('energy','एनर्जी'), ('emotions','एमोशन'),
    ('machine','मशीन'), ('tragedy','ट्राजेडी'), ('murder','मर्डर'),
    ('sensitive','सेंसिटिव'), ('instrument','इंस्ट्रूमेंट'),
    ('reasoning','रीज़निंग'), ('crack','क्रैक'), ('clear','क्लियर'),
]

print('QUICK TEST v4 -- checking English words stay Latin\n')
all_ok = True
for i, t in enumerate(tests, 1):
    out = convert_chunk(t)
    bad = [eng for eng, deva in BAD_TRANSLITERATIONS if deva in out]
    status = f'FAIL -- transliterated: {bad}' if bad else 'PASS'
    print(f'Test {i}: [{status}]')
    print(f'  IN : {t}')
    print(f'  OUT: {out}\n')
    if bad: all_ok = False

if all_ok:
    print('All tests passed!')
else:
    print('Some English words are still being transliterated.')
    print('Try reducing chunk_size to 200 for better accuracy.')

QUICK TEST v4 -- checking English words stay Latin

Test 1: [PASS]
  IN : Woh duniya ka sabse perfect reasoning machine tha, lekin emotions usse disturb karti thi.
  OUT: वो दुनिया का sabse perfect reasoning machine था, लेकिन emotions उससे disturb करती थी।

Test 2: [PASS]
  IN : Maine Atkinson brothers ki singular tragedy ke baare mein suna tha.
  OUT: मैंने Atkinson brothers की singular tragedy के बारे में सुना था.

Test 3: [PASS]
  IN : Holmes ke liye strong emotions uske sensitive instrument mein crack jaisi thi.
  OUT: Holmes के लिए strong emotions उसके sensitive instrument में crack जैसी थी।

Test 4: [PASS]
  IN : Sherlock Holmes ke liye woh hamesha THE woman rahi.
  OUT: Sherlock Holmes के लिए वो हमेशा THE woman रही.

All tests passed!


## Step 7 - Process Your Text

### Option A: Paste text directly

In [14]:
input_text = """
Sherlock Holmes ke liye woh hamesha THE woman rahi.
Maine usko kabhi aur kisi naam se mention karte nahi suna.
Yeh nahi tha ki usko Irene Adler se koi love feelings thi.
Woh duniya ka sabse perfect reasoning aur observing machine tha.
Par lover banne ke liye woh khud ko galat position mein daal deta.
Uske jaise nature mein strong emotions bahut disturbing hoti.
""".strip()

print('ORIGINAL:')
print(input_text)
print()
print('CONVERTING...')
converted_text = process_in_chunks(input_text)
print()
print('OUTPUT:')
print(converted_text)

ORIGINAL:
Sherlock Holmes ke liye woh hamesha THE woman rahi.
Maine usko kabhi aur kisi naam se mention karte nahi suna.
Yeh nahi tha ki usko Irene Adler se koi love feelings thi.
Woh duniya ka sabse perfect reasoning aur observing machine tha.
Par lover banne ke liye woh khud ko galat position mein daal deta.
Uske jaise nature mein strong emotions bahut disturbing hoti.

CONVERTING...
  Chunk 1 (301 chars)... done
  Chunk 2 (61 chars)... done
Done! 2 chunks processed.

OUTPUT:
Sherlock Holmes के लिए वो हमेशा THE woman रही।  
मैंने उसको कभी और किसी नाम से mention करते नहीं सुना।  
यह नहीं था कि उसको Irene Adler से कोई love feelings थी।  
वो दुनिया का सबसे perfect reasoning aur observing machine था।  
पर lover बनने के लिए वो खुद को गलत position में डाल देता।
उसके जैसे nature में strong emotions बहुत disturbing होती.


### Option B: Upload a .txt file

1.   List item
2.   List item



In [12]:
from google.colab import files

print('Upload your .txt file:')
uploaded = files.upload()

for filename, content in uploaded.items():
    print(f'File: {filename} ({len(content):,} bytes)')
    raw_text = content.decode('utf-8')
    print(f'Est. chunks: ~{len(raw_text) // 350 + 1}')
    print('Converting...\n')
    converted = process_in_chunks(raw_text, chunk_size=350)
    out_filename = f'converted_{filename}'
    with open(out_filename, 'w', encoding='utf-8') as f:
        f.write(converted)
    print(f'\nSaved: {out_filename}')
    print('PREVIEW (first 600 chars):')
    print(converted[:600])

Upload your .txt file:


Saving translation_hin_20260222_112302.txt to translation_hin_20260222_112302 (1).txt
File: translation_hin_20260222_112302 (1).txt (6,953 bytes)
Est. chunks: ~20
Converting...

  Chunk 1 (33 chars)... done
  Chunk 2 (3891 chars)... 

ReadTimeout: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=180)

### Option C: Download result

In [6]:
from google.colab import files

out_path = 'converted_output.txt'
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(converted)  # change to `converted` if used Option B
files.download(out_path)
print(f'Downloading: {out_path}')

NameError: name 'converted_text' is not defined

## Step 8 - Batch Process Multiple Files

In [ ]:
from google.colab import files
import os

print('Upload all .txt files:')
uploaded_files = files.upload()
batch_results = {}

for i, (filename, content) in enumerate(uploaded_files.items(), 1):
    print(f'\n[{i}/{len(uploaded_files)}] {filename}')
    try:
        raw = content.decode('utf-8')
        conv = process_in_chunks(raw, chunk_size=350)
        out = f'converted_{filename}'
        with open(out, 'w', encoding='utf-8') as f:
            f.write(conv)
        batch_results[filename] = ('OK', out)
    except Exception as e:
        batch_results[filename] = ('ERROR', str(e))
        print(f'Error: {e}')

print('\nSUMMARY')
for fname, (status, info) in batch_results.items():
    print(f'  [{status}] {fname} -> {info}')

for status, info in batch_results.values():
    if status == 'OK' and os.path.exists(info):
        files.download(info)
print('Done.')

## Step 9 - Quality Check & Diagnostics

In [ ]:
# Check for English words being wrongly transliterated
BAD_PAIRS = [
    ('perfect','परफेक्ट'), ('energy','एनर्जी'), ('emotions','एमोशन'),
    ('machine','मशीन'), ('tragedy','ट्राजेडी'), ('murder','मर्डर'),
    ('sensitive','सेंसिटिव'), ('instrument','इंस्ट्रूमेंट'),
    ('reasoning','रीज़निंग'), ('crack','क्रैक'), ('clear','क्लियर'),
    ('share','शेयर'), ('mission','मिशन'), ('family','फैमिली'),
    ('results','रिजल्ट'), ('nature','नेचर'), ('power','पावर'),
    ('admirable','एडमिरेबल'), ('distraction','डिस्ट्रैक्शन'),
    ('observer','ऑब्जर्वर'), ('action','एक्शन'), ('motive','मोटिव'),
]

def transliteration_check(converted):
    found = [(eng, deva) for eng, deva in BAD_PAIRS if deva in converted]
    if found:
        print(f'FAIL: {len(found)} English words transliterated to Devanagari:')
        for eng, deva in found:
            print(f'  {eng} was written as {deva}')
        print('\nTip: Reduce chunk_size to 200 for higher accuracy.')
    else:
        print('PASS: No wrongly transliterated English words detected!')

transliteration_check(converted_text)

In [ ]:
# Side-by-side comparison
def show_comparison(original, converted, max_lines=40):
    ol = original.split('\n')
    cl = converted.split('\n')
    n = min(max(len(ol), len(cl)), max_lines)
    print(f'{"ORIGINAL":<65} CONVERTED')
    print('-' * 130)
    for i in range(n):
        o = ol[i] if i < len(ol) else ''
        c = cl[i] if i < len(cl) else ''
        flag = ' [UNCHANGED]' if o.strip() and o == c else ''
        print(f'{o:<65} {c}{flag}')

show_comparison(input_text, converted_text)

In [ ]:
# Retry a specific sentence
problem = 'Woh duniya ka sabse perfect reasoning machine tha.'
print('IN :', problem)
print('OUT:', convert_chunk(problem, retry=True))

In [ ]:
# Auto-clean thinking bleed
leaking = [l for l in converted_text.split('\n')
           if l.strip() and any(re.match(p, l.strip(), re.IGNORECASE) for p in THINKING_PATTERNS)]
if leaking:
    print(f'Found {len(leaking)} leaked reasoning line(s):')
    for l in leaking: print(f'  {l[:100]}')
    converted_text = strip_reasoning_preamble(converted_text)
    print('Cleaned.')
else:
    print('No reasoning bleed detected.')

---
## Quick Reference

| Task | Cell |
|------|------|
| Paste and convert | Step 7 Option A |
| Upload .txt file | Step 7 Option B |
| Download result | Step 7 Option C |
| Multiple files | Step 8 |
| Check transliteration errors | Step 9 - transliteration_check |
| Side-by-side diff | Step 9 - show_comparison |
| Fix one sentence | Step 9 - Retry |

### What goes to Devanagari vs stays Latin

| Category | Script | Examples |
|----------|--------|----------|
| Hindi grammar words | Devanagari | है, में, तुम, वो, और, लेकिन, नहीं |
| Hindi-origin verbs | Devanagari | देखा, बोला, गया, लिया, चाहिए |
| English-origin descriptors | Latin | perfect, clear, sensitive, admirable |
| English-origin nouns | Latin | energy, tragedy, murder, mission, nature |
| English loanwords | Latin | coat, case, practice, doctor, police |
| Names & places | Latin | Holmes, Baker Street, London |

### If English words still getting transliterated
Try `chunk_size=200` for higher accuracy (slower but more precise).